# Эволюция схемы (Schema Evolution)

Одна из ключевых возможностей Iceberg — безопасное изменение схемы таблицы
**без переписывания уже записанных data-файлов**. Каждой колонке присвоен
внутренний числовой `field-id`, который не зависит от её имени или позиции —
именно поэтому переименование или изменение порядка колонок не требует
трогать данные: меняется только `metadata.json` (описание схемы), а не сами
parquet-файлы.

Поддерживаемые "безопасные" изменения:

- `ADD COLUMN` — добавить колонку (для старых строк вернётся `NULL`);
- `DROP COLUMN` — удалить колонку;
- `RENAME COLUMN` — переименовать колонку (по `field-id`, а не по имени);
- `ALTER COLUMN ... TYPE` — расширить тип (`int → bigint`, `float → double`,
  `decimal(p,s) → decimal(p', s)` с `p' >= p`), но не сузить;
- `ALTER COLUMN ... FIRST | AFTER` — изменить порядок колонок.

Важно: такие изменения **не создают новый снапшот данных** — меняется схема
таблицы, но история снапшотов (`.history`, `.snapshots`) не растёт.

In [1]:
from pyspark.sql import SparkSession

# Настраиваем каталог Iceberg прямо в коде, а не через spark-defaults.conf.
# Это даёт тот же результат, но позволяет менять параметры каталога
# независимо от конфигурации образа и делает ноутбук самодостаточным.
spark = (
    SparkSession.builder
    .appName("Iceberg basics")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.course", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.course.type", "rest")
    .config("spark.sql.catalog.course.uri", "http://rest-catalog:8181")
    .config("spark.sql.catalog.course.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.course.warehouse", "s3://warehouse")
    .config("spark.sql.catalog.course.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.course.s3.path-style-access", "true")
    .getOrCreate()
)

print("Spark + Iceberg готовы к работе")

Spark + Iceberg готовы к работе


Ниже определена вспомогательная функция `read_metadata_table` для чтения
служебных metadata-таблиц Iceberg в обход REST-каталога этого стенда —
подробное объяснение, зачем она нужна, см. в `00_Setup_and_Basics.ipynb`.
Вызовы `CALL course.system....` эту проблему не затрагивают и работают как
обычно — namespace `system` у них всегда одноуровневый.

In [11]:
from pyspark.sql import DataFrame


def read_metadata_table(spark, table_identifier, metadata_type):
    """Читает metadata-таблицу Iceberg (history, snapshots, files, partitions, manifests, ...)
    в обход REST-каталога, который отклоняет составные namespace в URL
    (см. подробное объяснение в 00_Setup_and_Basics.ipynb)."""
    jvm = spark._jvm
    table = jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, table_identifier)
    mtype = jvm.org.apache.iceberg.MetadataTableType.valueOf(metadata_type.upper())
    jdf = jvm.org.apache.iceberg.spark.SparkTableUtil.loadMetadataTable(spark._jsparkSession, table, mtype)
    return DataFrame(jdf, spark)

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS course.test")

spark.sql("DROP TABLE IF EXISTS course.test.schema_evolution")
spark.sql("""
CREATE TABLE course.test.schema_evolution (
    id      INT,
    name    STRING
)
USING iceberg
""")

spark.sql("INSERT INTO course.test.schema_evolution VALUES (1, 'Alice'), (2, 'Bob')")
spark.sql("SELECT * FROM course.test.schema_evolution ORDER BY id").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



## ADD COLUMN

Добавляем колонку `email`. У существующих строк она автоматически станет
`NULL` — физически данные не переписываются.

In [3]:
spark.sql("ALTER TABLE course.test.schema_evolution ADD COLUMN email STRING")
spark.sql("INSERT INTO course.test.schema_evolution VALUES (3, 'Carol', 'carol@mail.com')")

spark.sql("SELECT * FROM course.test.schema_evolution ORDER BY id").show(truncate=False)

+---+-----+--------------+
|id |name |email         |
+---+-----+--------------+
|1  |Alice|NULL          |
|2  |Bob  |NULL          |
|3  |Carol|carol@mail.com|
+---+-----+--------------+



## RENAME COLUMN

Переименование не трогает данные — под капотом меняется только имя,
привязанное к тому же `field-id`.

In [4]:
spark.sql("ALTER TABLE course.test.schema_evolution RENAME COLUMN email TO contact_email")
spark.sql("SELECT * FROM course.test.schema_evolution ORDER BY id").show(truncate=False)

+---+-----+--------------+
|id |name |contact_email |
+---+-----+--------------+
|1  |Alice|NULL          |
|2  |Bob  |NULL          |
|3  |Carol|carol@mail.com|
+---+-----+--------------+



## ALTER COLUMN TYPE (расширение типа)

`id` был `INT`, расширим его до `BIGINT`. Такое расширение всегда безопасно,
так как любое значение `int` помещается в `bigint`.

In [5]:
spark.sql("ALTER TABLE course.test.schema_evolution ALTER COLUMN id TYPE BIGINT")
spark.sql("DESCRIBE TABLE course.test.schema_evolution").show(truncate=False)

+-------------+---------+-------+
|col_name     |data_type|comment|
+-------------+---------+-------+
|id           |bigint   |NULL   |
|name         |string   |NULL   |
|contact_email|string   |NULL   |
+-------------+---------+-------+



## Изменение порядка колонок

Переносим `contact_email` сразу после `id`, до `name`.

In [6]:
spark.sql("ALTER TABLE course.test.schema_evolution ALTER COLUMN contact_email AFTER id")
spark.sql("SELECT * FROM course.test.schema_evolution ORDER BY id").show(truncate=False)

+---+--------------+-----+
|id |contact_email |name |
+---+--------------+-----+
|1  |NULL          |Alice|
|2  |NULL          |Bob  |
|3  |carol@mail.com|Carol|
+---+--------------+-----+



## DROP COLUMN

Удаляем колонку. Это тоже метаданная операция — старые parquet-файлы
физически продолжают содержать эту колонку на диске, но Iceberg перестаёт её
проецировать при чтении.

In [7]:
spark.sql("ALTER TABLE course.test.schema_evolution DROP COLUMN contact_email")
spark.sql("SELECT * FROM course.test.schema_evolution ORDER BY id").show(truncate=False)

+---+-----+
|id |name |
+---+-----+
|1  |Alice|
|2  |Bob  |
|3  |Carol|
+---+-----+



## Проверяем: изменения схемы не создают новых снапшотов данных

История снапшотов отражает только операции с данными (`INSERT`, `UPDATE`,
`DELETE`, `MERGE`, компакцию и т.п.), но не изменения схемы.

In [12]:
read_metadata_table(spark, "course.test.schema_evolution", "history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-17 09:58:35.218|4267121104820949487|NULL               |true               |
|2026-09-17 09:58:45.487|4350926833364341548|4267121104820949487|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [13]:
spark.stop()